# Assignment 2 · Part 3 — Sakila from a notebook
Connects to the `sakila` database on Cloud SQL with the Cloud SQL Python Connector.
Credentials come from environment variables (`CONNECTION_NAME`, `MYSQL_ROOT_PW`) — no password or project id in this notebook.

In [1]:
import os, warnings
import pandas as pd
from google.cloud.sql.connector import Connector

# pandas warns that it prefers SQLAlchemy; the pymysql connection works fine, so hide that one warning
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

connector = Connector()
conn = connector.connect(
    os.environ["CONNECTION_NAME"],          # project:region:instance, read from the environment
    "pymysql",
    user="root",
    password=os.environ["MYSQL_ROOT_PW"],   # read from env.sh, never typed into a cell
    db="sakila",
)

def q(sql):
    """Run one SQL statement and return the result as a DataFrame."""
    return pd.read_sql(sql, conn)

print("connected to sakila")

connected to sakila


## 2a · Tables, film count, actors named John
`SHOW FULL TABLES` lists every table with its type: Sakila has 16 base tables and 7 views.
`COUNT(*)` counts the rows in `film`. The actor query filters with `WHERE first_name = 'JOHN'`.

In [2]:
q("SHOW FULL TABLES")

,Tables_in_sakila,Table_type
0,actor,BASE TABLE
1,actor_info,VIEW
2,address,BASE TABLE
3,category,BASE TABLE
4,city,BASE TABLE
5,country,BASE TABLE
6,customer,BASE TABLE
7,customer_list,VIEW
8,film,BASE TABLE
9,film_actor,BASE TABLE


In [3]:
q("SELECT COUNT(*) AS films FROM film")

,films
0,1000


In [4]:
q("SELECT actor_id, first_name, last_name FROM actor WHERE first_name = 'JOHN'")

,actor_id,first_name,last_name
0,192,JOHN,SUVARI


## 2b · Films per category (JOIN + GROUP BY)
A film's category is stored in the link table `film_category`, so we JOIN `category` to `film_category`,
GROUP BY category and COUNT the films in each, largest first. Check: the counts must add up to 1,000.

In [5]:
per_cat = q("""
    SELECT c.name AS category, COUNT(*) AS films
    FROM category c
    JOIN film_category fc ON fc.category_id = c.category_id
    GROUP BY c.category_id, c.name
    ORDER BY films DESC
""")
assert per_cat.films.sum() == 1000, per_cat.films.sum()   # every film is in exactly one category
per_cat

,category,films
0,Sports,74
1,Foreign,73
2,Family,69
3,Documentary,68
4,Animation,66
5,Action,64
6,New,63
7,Drama,62
8,Games,61
9,Sci-Fi,61


## 2c · Customers with more than 40 rentals (GROUP BY + HAVING)
JOIN `customer` to `rental`, GROUP BY customer and COUNT their rentals.
`HAVING` filters after grouping (WHERE cannot see the count), keeping customers with more than 40.

In [6]:
q("""
    SELECT c.customer_id, c.first_name, c.last_name, COUNT(*) AS rentals
    FROM customer c
    JOIN rental r ON r.customer_id = c.customer_id
    GROUP BY c.customer_id, c.first_name, c.last_name
    HAVING rentals > 40
    ORDER BY rentals DESC
""")

,customer_id,first_name,last_name,rentals
0,148,ELEANOR,HUNT,46
1,526,KARL,SEAL,45
2,144,CLARA,SHAW,42
3,236,MARCIA,DEAN,42
4,75,TAMMY,SANDERS,41


## Part 4 · Top 5 customers by total payments, with the city of their store
Query written by Claude Code (Part 4.3); the joins, in my words:
- **customer → payment** (`p.customer_id = c.customer_id`): every payment row is attached to its customer, so `SUM(p.amount)` per customer is what they paid in total.
- **customer → store** (`s.store_id = c.store_id`): each customer has one home store, so this join adds no extra rows and does not change the totals.
- **store → address** (`a.address_id = s.address_id`): this is the *store's* address. Joining on `c.address_id` instead would give the customer's own city — the wrong answer.
- **address → city** (`ci.city_id = a.city_id`): turns the store's address into a city name.
Sorted by total paid (largest first), with `customer_id` as a tie-breaker so the order is the same every run.

In [7]:
top5 = q("""
    SELECT c.customer_id,
           CONCAT(c.first_name, ' ', c.last_name) AS customer,
           c.store_id,
           ci.city                                AS store_city,
           COUNT(p.payment_id)                    AS payments,
           ROUND(SUM(p.amount), 2)                AS total_paid
    FROM customer c
    JOIN payment p  ON p.customer_id = c.customer_id   -- customer -> payment
    JOIN store   s  ON s.store_id    = c.store_id      -- customer -> store
    JOIN address a  ON a.address_id  = s.address_id    -- store -> address
    JOIN city    ci ON ci.city_id    = a.city_id       -- address -> city
    GROUP BY c.customer_id, c.first_name, c.last_name, c.store_id, ci.city
    ORDER BY total_paid DESC, c.customer_id
    LIMIT 5
""")
top5

,customer_id,customer,store_id,store_city,payments,total_paid
0,526,KARL SEAL,2,Woodridge,45,221.55
1,148,ELEANOR HUNT,1,Lethbridge,46,216.54
2,144,CLARA SHAW,1,Lethbridge,42,195.58
3,137,RHONDA KENNEDY,2,Woodridge,39,194.61
4,178,MARION SNYDER,2,Woodridge,39,194.61


### Check — the total over all customers equals 67,406.56
The inner query uses the same joins as above but no `LIMIT`, so it totals every customer.
It is compared with `SUM(amount)` taken straight from `payment`: if the joins had dropped payments the sum would be lower, and if they had counted rows twice it would be higher.

In [8]:
check = q("""
    SELECT COUNT(*)                                    AS customers_with_payments,
           ROUND(SUM(t.total_paid), 2)                 AS sum_over_customers,
           (SELECT ROUND(SUM(amount), 2) FROM payment) AS payment_table_total
    FROM (
        SELECT c.customer_id, SUM(p.amount) AS total_paid
        FROM customer c
        JOIN payment p  ON p.customer_id = c.customer_id
        JOIN store   s  ON s.store_id    = c.store_id
        JOIN address a  ON a.address_id  = s.address_id
        JOIN city    ci ON ci.city_id    = a.city_id
        GROUP BY c.customer_id
    ) AS t
""")
check

,customers_with_payments,sum_over_customers,payment_table_total
0,599,67406.56,67406.56


In [9]:
conn.close(); connector.close(); print("closed")

closed
